# Lab Exercise: Detecting Outdated Orthophotos via AI Segmentation & Overture Maps

**📝 Scenario.** The City of Graz wants to systematically identify areas where their official orthophotos (e.g., from 2016-2018) are severely outdated due to recent urban development. Instead of manually scanning the whole city, they want an automated pipeline that compares **historical imagery** with **current map data** (Overture Maps) to flag "hotspots of change".

**🚚 What you will deliver**
1. A reproducible notebook that:
   - downloads historical orthophotos for a specific Region of Interest (ROI) via the ArcGIS REST API
   - queries current building footprints from **Overture Maps** using DuckDB
   - runs an AI foundation model (**SamGeo** / Segment Anything) to extract buildings from the old orthophoto
   - computes an **Inconsistency Analysis** (Area difference, Intersection over Union - IoU)
   - produces an interactive Plotly map highlighting the outdated areas

2. A short advisory brief (<500 words) answering:
   - **Top 3 sub-regions to flag** for new aerial surveying based on your metrics
   - **Technical risks** of using AI segmentation (SamGeo) for official building registries
   - **Ethical & Open Data considerations** when mixing Overture data with state-owned imagery


#### 👀 Engineering & Academic Quality
- **Correctness:** CRS handling (projected vs. geographic), valid geometries, exact IoU math
- **Engineering:** robust DuckDB queries, memory-efficient image processing, modular functions
- **Reasoning:** threshold choices for "inconsistency" are documented and justified
- **Communication:** map + brief clearly answer the client's problem

<div class="alert alert-warning">

#### 📃 Data sources & licences (you must always acknowledge them)
- **Overture Maps** data is licensed under the Community Data License Agreement (CDLA). [👀 see](https://overturemaps.org/)
- **Orthofotos Land Steiermark** are provided as Open Government Data (OGD) under CC BY 4.0. [👀 see](https://data.steiermark.at/)

---

## 🎯 Learning outcomes

---

1. **Cloud-Native Geoprocessing:** Query massive remote datasets (Overture GeoParquet) efficiently using `DuckDB` spatial extensions.
2. **GeoAI Application:** Apply Meta's Segment Anything Model (`SamGeo`) to georeferenced raster data to extract vector building footprints.
3. **Spatial Metrics:** Design and calculate computer vision metrics like **Intersection over Union (IoU)** using `GeoPandas`.
4. **CRS Mastery:** Seamlessly transition between web-mercator (image API), WGS84 (Overture/Plotly), and local projected CRS (metric area calculations).
5. **Interactive Visualization:** Overlay raster and vector data in a web-friendly `Plotly` map.


---

## 0. 🔧 Setup & Configuration

---


In [ ]:
# install uv
%pip install uv

In [ ]:
# installs for the notebook, uncomment to install them if not done already, or select the library you need to install
# %pip install geopandas duckdb shapely requests rasterio plotly segment-geospatial leafmap localtileserver geoai-py
!uv pip install geopandas duckdb shapely requests rasterio plotly segment-geospatial leafmap localtileserver geoai-py

In [1]:
#system
import os
from pathlib import Path

# Clear conflicting PROJ environment variables if they exist
# (I had to do this so that my leafmap works)
os.environ.pop('PROJ_LIB', None)
os.environ.pop('PROJ_DATA', None)

#data handling
import requests
import duckdb
import geopandas as gpd
import pandas as pd

#geo
import rasterio
from shapely.geometry import box
import shapely.wkt
import plotly.express as px
from samgeo import SamGeo

#visualization
import leafmap



In [2]:
DATA_DIR = Path("./data")
OUTPUT_DIR = Path("./outputs")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

CONFIG = {
    "crs_geographic": "EPSG:4326",   # WGS84
    "crs_projected": "EPSG:32633",   # UTM Zone 33N for metric area calculations
    "ortho_base_url": "https://gis.stmk.gv.at/image/rest/services/OGD_DOP/Flug_2016_2018_RGB/ImageServer/exportImage",
    
    #  CHOOSE YOUR AREA OF INTEREST (AOI)
    # 1. Go to http://bboxfinder.com
    # 2. Zoom to Graz and draw a SQUARE ☐ bounding box.
    # 3. RULE: The area should be ~25 Hectares!
    #    (If it is larger, the image will be downscaled and you will lose detail)
    # 4. Copy the coordinates from the bottom left (Format: min_lon, min_lat, max_lon, max_lat)
    
    # Paste your custom bounding box here:
    "bbox_custom": [None, None, None, None],

    # Example bounding boxes for different districts in Graz (you can use these if you don't want to draw your own)
    "bbox_waltendorf": [15.481281,47.061074,15.487804,47.065693],
    "bbox_mariatrost": [15.453300,47.091660,15.459416,47.095400],
    "bbox_liebenau": [15.464587,47.033865,15.470896,47.038515],
    "bbox_graz_center": [15.436778,47.067842,15.443537,47.072168]
}

# Decide which one to use for the rest of the script:
ACTIVE_BBOX = CONFIG["bbox_graz_center"]

<div class="alert alert-danger">     

**🚀 TODO**

1. Define a bounding box dictionary with `min_lon, min_lat, max_lon, max_lat` in WGS84.
2. by looking at the documentation and testing different combinations in the browser! (Hint: use the "Export Image" functionality in the ArcGIS REST API documentation to see which parameters are needed
3. Save the response as a `.tif` file in your `DATA_DIR`.

<div class="alert alert-info">

**💡 Python Tips: Dictionaries (Keys & Values)**

A Python **Dictionary** (`dict`) is used to store data values in **key-value pairs**. You can think of the **key** as the label or variable name, and the **value** as the actual data attached to it.

- **Syntax:** Dictionaries are written with curly brackets `{}`. The key and value are separated by a colon `:`, and each pair is separated by a comma.
    ```python
    my_settings = {
        "color": "blue",    # "color" is the key, "blue" is the value
        "opacity": 0.5
    }
    ```

- **Why use them for APIs?** When making HTTP requests, web servers expect specific parameter names. The Python `requests` library is smart: it takes your dictionary and automatically translates the keys and values into a URL query string (for example, `?color=blue&opacity=0.5`). 

- **Your Task:** In the code below, the *values* are already set. Your job is to replace the `?` with the exact *keys* (parameter names) that the ArcGIS API requires to understand our request!
</div>

In [8]:
# Orthophoto Download

#https://gis.stmk.gv.at/image/rest/services/OGD_DOP/Flug_2016_2018_RGB/ImageServer
#https://gis.stmk.gv.at/image/rest/services/OGD_DOP/Flug_2016_2018_RGB/ImageServer/exportImage?


def download_orthophoto(bbox: list, url: str, output_path: Path, size: str = "2000,2000") -> None: 
    """
    Downloads a historical orthophoto from the ArcGIS REST API.
    """
    # Convert the bbox list to a comma-separated string
    bbox_str = f"{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}" #students: f"{},{},{},{}"
    
    # Define the exact parameters required by the ArcGIS ImageServer -> let students figure out the parameters by looking at the documentation and testing different combinations in the browser! (Hint: use the "Export Image" functionality in the ArcGIS REST API documentation to see which parameters are needed)
    params = { #students!
        "bbox": bbox_str,
        "bboxSR": "4326",      # Explicitly state our bbox is in WGS84
        "size": size,          # Constrain image size to avoid server limits
        "format": "tiff",      # Request GeoTIFF format
        "f": "image"           # Request the actual image file, not JSON metadata
    }
    
    print(f"Requesting imagery from {url} with parameters: {params}")
    response = requests.get(url, params=params)
    
    # Raise an exception if the HTTP request failed (e.g., 404 or 400 error)
    response.raise_for_status()
    
    # Write the binary content to our data directory
    with open(output_path, "wb") as f:
        f.write(response.content)
        
    print(f"Orthophoto saved to: {output_path}")

# Execute the download for our Graz bounding box
ortho_file = DATA_DIR / "graz_ortho_2016_2018_liebenau.tif"
download_orthophoto(ACTIVE_BBOX, CONFIG["ortho_base_url"], ortho_file)

Requesting imagery from https://gis.stmk.gv.at/image/rest/services/OGD_DOP/Flug_2016_2018_RGB/ImageServer/exportImage with parameters: {'bbox': '15.464587,47.033865,15.470896,47.038515', 'bboxSR': '4326', 'size': '2000,2000', 'format': 'tiff', 'f': 'image'}
Orthophoto saved to: data\graz_ortho_2016_2018_liebenau.tif


### 1.3 Fixing the Server Distortion (Clipping with Rasterio)

Because we requested a fixed `2000,2000` pixel image, but your custom bounding box is likely not a perfect square, the ArcGIS server automatically expanded the geographical footprint of your image to make it square. 

If we don't fix this, our AI will analyze areas outside of our target zone!

To solve this, we will use `rasterio` (the standard Python library for raster data) and `shapely` (for geometry) to cleanly "clip" the downloaded GeoTIFF exactly to your original `ACTIVE_BBOX` coordinates.

In [9]:
import rasterio
from rasterio.mask import mask
from shapely.geometry import box
import geopandas as gpd

# --- HELPER FUNCTION ---
def clip_raster_to_bbox(input_raster_path, output_raster_path, bbox_wgs84):
    """
    Clips a raster image using a WGS84 bounding box and saves the result.
    This hides the complex metadata management from the main script!
    """
    # 1. Create the Polygon from our bounding box
    min_lon, min_lat, max_lon, max_lat = bbox_wgs84
    bbox_polygon = box(min_lon, min_lat, max_lon, max_lat)
    
    # 2. Put it in a GeoDataFrame
    gdf_bbox = gpd.GeoDataFrame({'geometry': [bbox_polygon]}, crs="EPSG:4326")
    
    # 3. Open the image, match the Coordinate Systems, and cut!
    with rasterio.open(input_raster_path) as src:
        # Reproject our polygon to match the image's exact CRS
        gdf_bbox_projected = gdf_bbox.to_crs(src.crs)
        projected_polygon = gdf_bbox_projected.geometry.iloc[0]
        
        # Cut the image (returns the new pixels and the new spatial mapping)
        out_image, out_transform = mask(src, [projected_polygon], crop=True)
        
        # Copy old metadata and update it with the new dimensions
        out_meta = src.meta.copy()
        out_meta.update({
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform
        })

    # 4. Save the cleanly clipped image
    with rasterio.open(output_raster_path, "w", **out_meta) as dest:
        dest.write(out_image)


# --- MAIN SCRIPT  ---

clipped_ortho_file = DATA_DIR / "graz_ortho_clipped_liebenau.tif"

# We call our function with our inputs
clip_raster_to_bbox(
    input_raster_path=ortho_file,
    output_raster_path=clipped_ortho_file,
    bbox_wgs84=ACTIVE_BBOX
)

print(f"Clipped image saved to: {clipped_ortho_file}")

Clipped image saved to: data\graz_ortho_clipped_liebenau.tif


---

## 2. 🏢 Query Current Buildings via DuckDB (Overture Maps)

---
Overture Maps distributes its data as Cloud-Native GeoParquet on Amazon S3. Instead of downloading the whole world, we use **DuckDB**'s spatial extension to push the bounding box filter directly to the cloud and only download what we need.

<div class="alert alert-danger">     

**🚀 TODO**

1. Install and load the extensions in DuckDB (look at the documentation).
2. Write a SQL query to select `id`, `names.primary`, and the geometry from the Overture S3 bucket.
3. Filter the query using your `bbox`.
4. Convert the DuckDB result into a `GeoDataFrame`.

In [10]:
# Fetching Overture Maps Data via DuckDB
# Could change to dynamic path based on release versions, but for simplicity we stick
# to a known release from the documentation?

#https://docs.overturemaps.org/getting-data/
#https://docs.overturemaps.org/schema/reference/buildings/building/

def fetch_overture_buildings(bbox: list, crs: str) -> gpd.GeoDataFrame:

    min_lon, min_lat, max_lon, max_lat = bbox #students!
    
    # 1. Connect to DuckDB and load extensions
    con = duckdb.connect()

    con.execute("INSTALL spatial; LOAD spatial;") #students
    con.execute("INSTALL httpfs; LOAD httpfs;") #students
    con.execute("SET s3_region='us-west-2';") #students
    
    
    # 2. Write the SQL query
    # Using the release from the documentation: 2026-02-18.0 #students?
    query = f""" 
    SELECT
        id,
        type,
        names.primary AS name,
        ST_AsText(geometry) AS geometry
    FROM read_parquet('s3://overturemaps-us-west-2/release/2026-02-18.0/theme=buildings/type=building/*', hive_partitioning=1)
    WHERE bbox.xmin >= {min_lon}
      AND bbox.xmax <= {max_lon}
      AND bbox.ymin >= {min_lat}
      AND bbox.ymax <= {max_lat};
    """
    
    # 3. Execute query and fetch as DataFrame
    df = con.execute(query).df()
    
    # 4. Convert WKT to Shapely geometries and build GeoDataFrame
    df['geometry'] = df['geometry'].apply(shapely.wkt.loads)
    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs=crs)
    
    return gdf

# Execute the function for our active bounding box
gdf_overture = fetch_overture_buildings(ACTIVE_BBOX, CONFIG["crs_geographic"])
print(f"Loaded {len(gdf_overture)} buildings from Overture Maps.")


overture_file = DATA_DIR / "overture_buildings_liebenau.gpkg"
gdf_overture.to_file(overture_file, driver="GPKG")

display(gdf_overture.head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 292 buildings from Overture Maps.


,id,type,name,geometry
0,985f0f1b-b1e4-48ec-8a62-9a7d324e1ef9,building,None,"POLYGON ((15.46904 47.03432, 15.46871 47.03422..."
1,8520b184-b312-4932-bde6-35b8429c0e76,building,None,"POLYGON ((15.46875 47.03433, 15.46866 47.03431..."
2,a3e93c88-9c35-47e9-b665-3b4309e7cfd7,building,None,"POLYGON ((15.46872 47.03438, 15.46863 47.03435..."
3,9780f86f-6adb-4912-8efb-336ec62bacaa,building,None,"POLYGON ((15.46967 47.03458, 15.46933 47.03449..."
4,7c5ab5f4-0ded-420b-8212-2be0a49db531,building,None,"POLYGON ((15.46996 47.03408, 15.47012 47.03411..."


In [ ]:
''' 
VISUALIZATION OF OVERTURE BUILDINGS ON TOP OF ORTHOPHOTO (Leafmap)
TODO: Try with plotly
'''

min_lon, min_lat, max_lon, max_lat = ACTIVE_BBOX
center_lat = (min_lat + max_lat) / 2
center_lon = (min_lon + max_lon) / 2

# Initialize the map
m = leafmap.Map(center=[center_lat, center_lon], zoom=16, height="600px")
m.clear_layers()

# Add our local, clipped orthophoto
ortho_path = str(DATA_DIR / "graz_ortho_clipped_waltendorf.tif") 
m.add_raster(ortho_path, layer_name="Orthophoto Waltendorf(2016-2018)", opacity=0.7)

# 2. Add the Overture buildings
m.add_gdf(
    gdf_overture, 
    layer_name="Overture Buildings (2026)", 
    fill_colors=["#00a8ff"],
    color="blue",
    weight=1
)

m

Map(center=[47.0633835, 15.4845425], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title…

---

## 3AI Segmentation (GeoAI with SamGeo)

---

Now we enter the Machine Learning part of the pipeline. We have an old orthophoto (Raster) and we need building footprints (Vector). 



We will use the **Segment Anything Model (SAM)** developed by Meta. SAM is a "Foundation Model" for computer vision. It wasn't explicitly trained just for buildings; it was trained on billions of images to understand the concept of "objects" and their boundaries. 

To bridge the gap between pure computer vision and GIS, we use the `segment-geospatial` (`samgeo`) Python library. It handles the spatial referencing, splits the large orthophoto into smaller chunks for the AI, and translates the pixel-based predictions back into georeferenced polygons.

<div class="alert alert-warning">

**⚠️ Hardware & Performance Note:**
SAM is a large Deep Learning model. Since most of you will be running this on laptops without a dedicated Nvidia GPU, we will use the smaller `"vit_b"` (Vision Transformer Base) model. 

Also, SAM operates in `automatic=True` mode here (Automatic Mask Generation). This means SAM does not know *what* a building is. It will segment **everything**—houses, cars, trees, and shadows. We will filter out the non-buildings in the next step using our Overture ground-truth!
</div>

In [3]:
# The SamGEO Segmentation
#https://samgeo.gishub.org/examples/automatic_mask_generator/


# Paths for our input and outputs
ortho_file = DATA_DIR / "graz_ortho_clipped.tif"
mask_raster_file = OUTPUT_DIR / "sam_masks_center_b.tif"
vector_file = OUTPUT_DIR / "sam_polygons_center_b.gpkg"

# Initialize the Segment Anything Model
# We use 'vit_b' to save memory 
# but 'vit_l' or 'vit_h' would give more detailed masks at the cost of more memory and processing time
# vit_b = 300mb, vit_l = 1.2gb, vit_h = 3.2gb
# runtime vit_h/64pps on strong GPU: 6 min

sam = SamGeo(
    model_type="vit_b",
    automatic=True, # Automatically generate masks for the whole image
    sam_kwargs={"points_per_side": 64, "min_mask_region_area": 20} # Adjust density of sampling points (lower = faster, but less detailed)
)

# 2. Generate raster masks from the orthophoto
print("Running AI segmentation on the orthophoto (This may take a few minutes!)")
sam.generate(
    source=str(ortho_file),
    output=str(mask_raster_file)
)

# Convert the raster masks into vector polygons
sam.tiff_to_vector(str(mask_raster_file), str(vector_file))

# Load the generated polygons into a GeoDataFrame and clean them
gdf_samgeo = gpd.read_file(vector_file)

# Basic Geometry Cleaning: Remove invalid or empty shapes that the AI might have produced
gdf_samgeo = gdf_samgeo[gdf_samgeo.geometry.is_valid]
gdf_samgeo = gdf_samgeo[~gdf_samgeo.geometry.is_empty]

# The tiff_to_vector function sometimes loses the CRS, so we explicitly assign it
# The orthophoto we downloaded via ArcGIS was requested in WGS84, so the output is WGS84
gdf_samgeo.set_crs(CONFIG["crs_geographic"], inplace=True, allow_override=True)

print(f"SAM generated: {len(gdf_samgeo)} valid polygons.")

# Display the first few rows
display(gdf_samgeo.head())

Running AI segmentation on the orthophoto (This may take a few minutes!)
SAM generated: 155 valid polygons.


,value,geometry
0,92.0,"POLYGON ((533555.16018 5213271.51045, 533555.1..."
1,92.0,"POLYGON ((533554.64429 5213270.99457, 533554.6..."
2,92.0,"POLYGON ((533554.1284 5213269.4469, 533554.128..."
3,92.0,"POLYGON ((533554.38635 5213267.64129, 533554.3..."
4,92.0,"POLYGON ((533554.64429 5213265.83568, 533554.6..."


In [6]:
ortho_file = str(DATA_DIR / "graz_ortho_clipped.tif")
vector_file = str(OUTPUT_DIR / "sam_polygons_centre_h.gpkg")

# 2. Load the SAM polygons
gdf_samgeo = gpd.read_file(vector_file)

# Calculate the center of our bounding box to focus the map
min_lon, min_lat, max_lon, max_lat = ACTIVE_BBOX
center_lat = (min_lat + max_lat) / 2
center_lon = (min_lon + max_lon) / 2


# 3. Initialize the map
m = leafmap.Map(center=[center_lat, center_lon], zoom=16, height="650px")
m.clear_layers()

# 4. Add the clipped orthophoto as the basemap
m.add_raster(ortho_file, layer_name="Historical Orthophoto (2016)")

# 5. Add the SAM AI Polygons on top
m.add_gdf(
    gdf_samgeo,
    layer_name="SAM AI Polygons",
    fill_colors=["#ff00ff"],
    color="magenta",
    weight=2
)

# Display the map
m

Map(center=[47.070005, 15.440158], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title',…

In [12]:
# GEOAI Segmentation
# https://opengeoai.org/examples/building_footprints_usa/#visualize-building-footprints

import geoai
import geopandas as gpd

ortho_path = str(DATA_DIR / "graz_ortho_clipped.tif")
ai_buildings_path = str(OUTPUT_DIR / "geoai_buildings_centre.geojson")

# Initialize the building footprint extraction model
print("Initializing the GeoAI Building Extractor...")
extractor = geoai.BuildingFootprintExtractor()

# Extract building footprints directly to vector (Option 2 from docs)
print("Analyzing the 2016 orthophoto and extracting buildings... (This may take a minute)")
gdf_ai = extractor.process_raster(
    ortho_path,
    output_path=ai_buildings_path,
    batch_size=4,
    confidence_threshold=0.5,
    overlap=0.25,
    nms_iou_threshold=0.5,
    min_object_area=50,
    mask_threshold=0.5,
    simplify_tolerance=1.0,
)

print(f"Found {len(gdf_ai)} raw building footprints")

# Regularize the polygons (Square off the corners)

gdf_ai_reg = extractor.regularize_buildings(
    gdf=gdf_ai,
    min_area=50,
    angle_threshold=15,
    orthogonality_threshold=0.3,
    rectangularity_threshold=0.7,
)

# Ensure the CRS matches our WGS84 standard
gdf_ai_reg.set_crs(CONFIG["crs_geographic"], inplace=True, allow_override=True)

print(f"Regularized {len(gdf_ai_reg)} buildings")
display(gdf_ai_reg.head())

Initializing the GeoAI Building Extractor...
Model path not specified, downloading from Hugging Face...
Model downloaded to: C:\Users\Timon\.cache\huggingface\hub\models--giswqs--geoai\snapshots\089548329c81f128fa12576663e7abdedb5cfa0e\building_footprints_usa.pth
Model loaded successfully
Analyzing the 2016 orthophoto and extracting buildings... (This may take a minute)
Processing with parameters:
- Confidence threshold: 0.5
- Tile overlap: 0.25
- Chip size: (512, 512)
- NMS IoU threshold: 0.5
- Mask threshold: 0.5
- Min object area: 50
- Max object area: None
- Simplify tolerance: 1.0
- Filter edge objects: True
- Edge buffer size: 20 pixels
Dataset initialized with 5 rows and 6 columns of chips
Image dimensions: 2000 x 1876 pixels
Chip size: 512 x 512 pixels
Overlap: 25.0% (stride_x=384, stride_y=384)
CRS: EPSG:32633
Processing raster with 8 batches


100%|██████████| 8/8 [00:39<00:00,  4.88s/it]


Objects before filtering: 193
Objects after filtering: 184
Saved 184 objects to outputs\geoai_buildings_centre.geojson
Found 184 raw building footprints
Regularizing 184 objects...
- Angle threshold: 15° from 90°
- Min orthogonality: 30.0% of angles
- Min rectangularity: 70.0% of bounding box area


100%|██████████| 184/184 [00:00<00:00, 4467.24it/s]

Regularization completed:
- Total objects: 184
- Rectangular objects: 6 (3.3%)
- Other regularized objects: 0 (0.0%)
- Unmodified objects: 178 (96.7%)
Regularized 184 buildings


,geometry,confidence,class,regularized
259,"POLYGON ((533588.43498 5212895.42785, 533587.4...",0.999458,1,NaN
67,"POLYGON ((533246.401 5213179.68233, 533246.143...",0.999229,1,NaN
0,"POLYGON ((533247.43278 5213172.71784, 533247.1...",0.998377,1,NaN
68,"POLYGON ((533242.78979 5213053.28968, 533242.5...",0.998292,1,NaN
244,"POLYGON ((533576.3116 5212910.64656, 533575.53...",0.997793,1,NaN


In [7]:


ortho_path = str(DATA_DIR / "graz_ortho_clipped.tif")
geoai_file = str(OUTPUT_DIR / "geoai_buildings_centre.geojson")

# 2. Load the GeoAI polygons
gdf_geoai_map = gpd.read_file(geoai_file)

# Calculate the center of our bounding box to focus the map
min_lon, min_lat, max_lon, max_lat = ACTIVE_BBOX
center_lat = (min_lat + max_lat) / 2
center_lon = (min_lon + max_lon) / 2


# 3. Initialize the map
m_geoai = leafmap.Map(center=[center_lat, center_lon], zoom=16, height="650px")
m_geoai.clear_layers()

# 4. Add the clipped historical orthophoto as the basemap
m_geoai.add_raster(ortho_path, layer_name="Historical Orthophoto (2016)")

# 5. Add the GeoAI Polygons on top
m_geoai.add_gdf(
    gdf_geoai_map,
    layer_name="GeoAI Regularized Buildings",
    fill_colors=["#ffaa00"], # A nice visible orange/gold
    color="darkorange",
    weight=2
)

# Display the map
m_geoai

Map(center=[47.070005, 15.440158], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title',…

### 4. 🏙️ Urban Change Detection: SAMGeo vs. Modern Reality (Overture)

We will overlay our generated 2016 buildings (SAMGeo, Magenta) with the modern 2026 ground truth (Overture Maps, Blue). 

<div class="alert alert-info">

**💡 Visual Interpretation:**
* **Perfect Match (Purple-ish):** SAM correctly identified a building that is still there today.
* **Blue only:** This is likely a **NEW** building constructed between 2016 and 2026!
* **Magenta only:** Either SAM made a mistake (false alarm), or a building was demolished.
</div>

In [ ]:
import plotly.express as px
import pandas as pd
import geopandas as gpd

print("Preparing data for the final visual comparison...")

# 1. Load the data 
samgeo_polygons_file = OUTPUT_DIR / "sam_polygons_mariatrost_h.gpkg" # Ersetze mit deinem Dateinamen aus Step 6
overture_file = DATA_DIR / "overture_buildings_mariatrost.gpkg" # Ersetze mit deinem Dateinamen aus Step 4

gdf_sam = gpd.read_file(samgeo_polygons_file)
gdf_overture = gpd.read_file(overture_file)

# 2. Reprojection
# WGS84 ist zwingend erforderlich für Plotly Webmaps
gdf_sam_wgs = gdf_sam.to_crs("EPSG:4326")      
gdf_overture_wgs = gdf_overture.to_crs("EPSG:4326") 

# Add identification labels so Plotly knows which polygon belongs to which era
gdf_sam_wgs['Source'] = 'SAM GeoAI'
gdf_overture_wgs['Source'] = 'Overture'

# 3. Combine the Datasets into one large table
# Nutze pd.concat, um die beiden GeoDataFrames untereinander zu hängen
gdf_combined = pd.concat([gdf_sam_wgs, gdf_overture_wgs], ignore_index=True)

print("Building Plotly Map...")
# Calculate center for the camera
center_lat = gdf_combined.geometry.centroid.y.mean()
center_lon = gdf_combined.geometry.centroid.x.mean()

# 4. Configure Plotly
fig = px.choropleth_mapbox(
    data_frame=gdf_combined,                   
    geojson=gdf_combined.geometry,                      
    locations=gdf_combined.index,      # Use the DataFrame index to match geometries
    color='Source',                    # Color the polygons based on the Source column               
    color_discrete_map={
        'Overture': '#00a8ff', # Bright Blue
        'SAM GeoAI': '#ff00ff'    # Magenta
    },
    center={"lat": center_lat, "lon": center_lon},
    mapbox_style="carto-darkmatter",
    zoom=15,
    opacity=0.6,
    hover_name='Source'
)

# Clean up layout
fig.update_layout(
    margin={"r":0,"t":40,"l":0,"b":0},
    title_text="Urban Development Comparison (Click Legend to Toggle Layers)",
    legend_title_text="Datasets"
)

# Show the map
fig.show()

### Spatial Metrics

To evaluate how well our AI (SAMGeo) performed in 2016 compared to the modern reality (Overture 2026), we use **Area-Based Binary Classification Metrics**. Since we are dealing with spatial data, we don't just count *entire* buildings; we measure the actual overlapping *square meters*.

Here are the key variables we just calculated:

* **Total Area (`sam_total_area` & `ov_total_area`):** The absolute sum of all building footprints in square meters for each respective dataset.
* **Intersection Area:** The literal overlapping space where a 2016 SAM building and a 2026 Overture building sit on the exact same piece of land.

From these three core areas, we derive our performance scores:

**1. Precision (Präzision)**
* **The Question:** *"Out of all the pixels SAM claimed were buildings, how many were ACTUALLY buildings?"*
* **The Formula:** $Precision = \frac{Intersection Area}{SAM Total Area}$
* **Interpretation:** A low precision means the AI had a lot of "False Positives" 

**2. Recall (Sensitivität)**
* **The Question:** *"Out of all the real buildings that exist in Overture, how many did SAM successfully find?"*
* **The Formula:** $Recall = \frac{Intersection Area}{Overture Total Area}$
* **Interpretation:** A low recall means the AI had many "False Negatives"

**3. F1-Score**
* **The Question:** *"What is the overall reliability of the model?"*
* **The Formula:** $F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$
* **Interpretation:** The F1-Score is the harmonic mean of Precision and Recall.

In [ ]:
import geopandas as gpd
import pandas as pd

print("Calculating quantitative metrics (SAMGeo vs. Overture)...")

gdf_sam = gpd.read_file(OUTPUT_DIR / "sam_polygons_mariatrost_b.gpkg")
gdf_overture = gpd.read_file(DATA_DIR / "overture_buildings_mariatrost.gpkg")


# TASK 1: Reprojection
# Use the CONFIG dictionary to pass the projected CRS (UTM Zone 33N)
gdf_sam_utm = gdf_sam.to_crs(CONFIG["crs_projected"])       
gdf_ov_utm = gdf_overture.to_crs(CONFIG["crs_projected"])   

# TASK 2: Calculate Total Area
# Calculate the area for the geometries and sum them up
sam_total_area = gdf_sam_utm.geometry.area.sum() 
ov_total_area = gdf_ov_utm.geometry.area.sum()  

# Calculate the Intersection (Overlap) Area (Pre-filled for you)
# gpd.overlay does a precise polygon-on-polygon intersection
intersection = gpd.overlay(gdf_sam_utm, gdf_ov_utm, how='intersection')
intersection_area = intersection.geometry.area.sum()

# Calculate Binary Classification Metrics
precision = intersection_area / sam_total_area if sam_total_area > 0 else 0
recall = intersection_area / ov_total_area if ov_total_area > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# TASK 3: Calculate Centroids

sam_centroids = gdf_sam_utm.copy()
overture_centroids = gdf_ov_utm.copy()

# Replace the geometry of the polygons with their exact center points
sam_centroids['geometry'] = gdf_sam_utm.centroid  
overture_centroids['geometry'] = gdf_ov_utm.centroid    

# Find the nearest Overture centroid for each SAM centroid
nearest_points = gpd.sjoin_nearest(sam_centroids, overture_centroids, distance_col="distance")


# TASK 4: Calculate Mean Distance
# Calculate the average (mean) of the resulting distance column
mean_centroid_dist = nearest_points['distance'].mean() 


# Display Results
print("-" * 50)
print(f"📊 SAMGeo Performance Metrics (2016 vs 2026)")
print("-" * 50)
print(f"Total SAMGeo Area (2016):    {sam_total_area:,.1f} m²")
print(f"Total Overture Area (2026):  {ov_total_area:,.1f} m²")
print(f"Intersection Area (Overlap): {intersection_area:,.1f} m²")
print("-" * 50)
print(f"Precision:                   {precision:.3f}  (Are predictions actually buildings?)")
print(f"Recall:                      {recall:.3f}  (Did we find all buildings?)")
print(f"F1-Score (Overall):          {f1_score:.3f}")
print("-" * 50)
print(f"Mean Centroid Shift:         {mean_centroid_dist:.2f} meters")
print("-" * 50)

Calculating quantitative metrics (SAMGeo vs. Overture)...
----------------------------------------
📊 SAMGeo Performance Metrics (Against 2026 Overture Truth)
----------------------------------------
Total SAMGeo Area (2016): 145,161.9 m²
Total Overture Area (2026): 110,382.7 m²
Intersection Area (Overlap): 76,248.9 m²
----------------------------------------
Precision:                0.525  (Are predictions actually buildings?)
Recall:                   0.691  (Did we find all buildings?)
F1-Score (Overall):       0.597
----------------------------------------


---

## Optional: SAM3 Segmentation Model

---

In [ ]:
import torch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from transformers import Sam3Processor, Sam3Model
from huggingface_hub import login
from pathlib import Path


# https://samgeo.gishub.org/examples/sam3_image_segmentation/
# requires hugging face account, wait for approval, generate access token, paste token without commiting to github...


# --- CONFIGURATION ---
HF_TOKEN = ""
ORTHO_PATH = "./data/graz_ortho_clipped_mariatrost.tif" # Adjust path if necessary
TEXT_PROMPT = "building"

# 1. Login to Hugging Face
print("Logging into Hugging Face...")
login(token=HF_TOKEN)

# 2. Hardware check (CUDA for NVIDIA, MPS for Mac, otherwise CPU)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
    
print(f"Loading SAM 3 on device: {device}... (This takes a while the first time due to the download!)")

# 3. Load the Foundation Model and the Processor
model = Sam3Model.from_pretrained("facebook/sam3").to(device)
processor = Sam3Processor.from_pretrained("facebook/sam3")
print("SAM 3 loaded successfully!")

# 4. Load image (lift PIL safety limit for large TIFs)
Image.MAX_IMAGE_PIXELS = None 
image = Image.open(ORTHO_PATH).convert("RGB")

print(f"\nStarting Open-Vocabulary search for: '{TEXT_PROMPT}'...")

# 5. Send prompts to the model
inputs = processor(images=image, text=TEXT_PROMPT, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

# 6. Convert raw tensors into clean masks
results = processor.post_process_instance_segmentation(
    outputs,
    threshold=0.25,        # Confidence Threshold
    mask_threshold=0.5,    # Pixel Threshold
    target_sizes=inputs.get("original_sizes").tolist()
)[0]

found_objects = len(results['masks'])
print(f"SAM 3 found {found_objects} objects of type '{TEXT_PROMPT}'.")

# --- VISUALIZATION ---
def overlay_masks(base_image, output_masks):
    """Overlays the found masks semi-transparently on the original image."""
    base_image = base_image.convert("RGBA")
    
    # Move masks to CPU and convert to Numpy arrays
    masks_np = 255 * output_masks.cpu().numpy().astype(np.uint8)
    n_masks = masks_np.shape[0]
    
    if n_masks == 0:
        return base_image
        
    # Generate a rainbow of colors for the found instances
    cmap = matplotlib.colormaps.get_cmap("rainbow").resampled(n_masks)
    colors = [tuple(int(c * 255) for c in cmap(i)[:3]) for i in range(n_masks)]

    # Paint each mask individually onto the image
    for mask_array, color in zip(masks_np, colors):
        mask_img = Image.fromarray(mask_array)
        overlay = Image.new("RGBA", base_image.size, color + (0,))
        
        # Set transparency level (Alpha)
        alpha = mask_img.point(lambda v: int(v * 0.5))
        overlay.putalpha(alpha)
        
        # Blend image and mask
        base_image = Image.alpha_composite(base_image, overlay)
        
    return base_image

if found_objects > 0:
    print("Rendering the final image...")
    composite_image = overlay_masks(image, results["masks"])

    plt.figure(figsize=(15, 15))
    plt.imshow(composite_image)
    plt.axis('off')
    plt.title(f"SAM 3 Results for text prompt: '{TEXT_PROMPT}'", fontsize=18, pad=20)
    plt.tight_layout()
    plt.show()
else:
    print("No objects found matching this prompt.")

In [3]:
import rasterio
from rasterio.features import shapes
from shapely.geometry import shape
import geopandas as gpd

print("Vectorizing SAM 3 masks into geospatial polygons...")

# 1. Get the spatial metadata from our clipped orthophoto
# We need this to know exactly where pixel (0,0) is in the real world
with rasterio.open(ORTHO_PATH) as src:
    ortho_transform = src.transform
    ortho_crs = src.crs

# 2. Extract the raw masks from the SAM 3 results
# We move them to the CPU, convert to NumPy, and make them standard 8-bit integers (0 or 1)
masks_np = results["masks"].cpu().numpy().astype('uint8')

# 3. Convert Pixels to Polygons
extracted_polygons = []

# SAM 3 returns an array of masks (one for each object it found)
for single_mask in masks_np:
    # The 'shapes' function traces the outline of contiguous pixels.
    # We only want to trace where the mask is 1 (the object), not the 0s (background)
    for geom, value in shapes(single_mask, mask=(single_mask == 1), transform=ortho_transform):
        # Convert the raw GeoJSON-like geometry into a standard Shapely Polygon
        extracted_polygons.append(shape(geom))

# 4. Create a GeoDataFrame from our new polygons
gdf_sam3_buildings = gpd.GeoDataFrame({'geometry': extracted_polygons}, crs=ortho_crs)

# 5. Optional but highly recommended: Simplify the geometry
# Raw pixel-tracing creates "staircase" edges (like Minecraft). 
# We simplify the lines slightly (tolerance in meters) to get cleaner building shapes.
gdf_sam3_buildings['geometry'] = gdf_sam3_buildings.geometry.simplify(0.3)

# 6. Save to disk so we can compare it with Overture!
output_file = "./data/sam3_buildings_2016_mariatrost.gpkg"
gdf_sam3_buildings.to_file(output_file, driver="GPKG")

print(f"Success! Saved {len(gdf_sam3_buildings)} building polygons to: {output_file}")

# Let's peek at the data table we just created
gdf_sam3_buildings.head()

Vectorizing SAM 3 masks into geospatial polygons...
Success! Saved 178 building polygons to: ./data/sam3_buildings_2016_mariatrost.gpkg


,geometry
0,"POLYGON ((534565.216 5215472.946, 534562.883 5..."
1,"POLYGON ((534826.036 5215776.459, 534825.336 5..."
2,"POLYGON ((534814.372 5215796.522, 534812.738 5..."
3,"POLYGON ((534586.445 5215538.268, 534584.812 5..."
4,"POLYGON ((534610.474 5215499.075, 534607.441 5..."
